In [6]:
"""Préparation des données UNSW-NB15 (projet HyperIDS).

Reproduit le notebook 02_preparation. Usage :

    train, val, test, x_cols = load_prepared()
    verify(train, val, test)

Sur Kaggle, copier le contenu de ce fichier dans la première cellule de code
du notebook (les notebooks n'importent pas src/ directement).
Les noms des fichiers du dataset sont inversés dans la version Kaggle utilisée.
"""
import hashlib

import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
BASE = '/kaggle/input/datasets/mrwellsdavid/unsw-nb15/'
TRAIN_FILE = 'UNSW_NB15_testing-set.csv'   # 175 341 lignes : entraînement officiel
TEST_FILE = 'UNSW_NB15_training-set.csv'   # 82 332 lignes : test officiel

TARGET = 'label'
DROP_COLS = ['id', 'attack_cat', 'ct_ftp_cmd']
CAT_COLS = ['proto', 'service', 'state']
MIN_COUNT = 100
VAL_SIZE = 0.2

EXPECTED = {'train': 'fb8d73b1cdba', 'val': 'f5db41df66c1', 'test': '0f96e3a31bcc'}


def _clean(df, feats):
    """Retire les groupes à étiquettes contradictoires, puis les doublons."""
    n_labels = df.groupby(feats)[TARGET].transform('nunique')
    df = df[n_labels == 1]
    df = df.drop_duplicates(subset=feats, keep='first')
    return df.reset_index(drop=True)


def _encode(df):
    df = df.drop(columns=['id', 'ct_ftp_cmd'])
    return pd.get_dummies(df, columns=CAT_COLS, dtype=int)


def load_prepared(base=BASE):
    """Retourne (train, val, test, x_cols). attack_cat reste dans les tableaux
    mais ne fait pas partie des caractéristiques (x_cols)."""
    train_raw = pd.read_csv(base + TRAIN_FILE)
    test_raw = pd.read_csv(base + TEST_FILE)
    feats = [c for c in train_raw.columns if c not in DROP_COLS + [TARGET]]

    train = _clean(train_raw, feats)
    test = _clean(test_raw, feats)

    # retire du test les lignes déjà présentes dans l'entraînement
    m = test.merge(train[feats].drop_duplicates(), on=feats, how='left', indicator=True)
    assert len(m) == len(test)
    overlap = (m['_merge'] == 'both').to_numpy()
    test = test[~overlap].reset_index(drop=True)

    # découpage entraînement / validation, stratifié sur attack_cat
    train_fit, val = train_test_split(
        train, test_size=VAL_SIZE, stratify=train['attack_cat'], random_state=SEED
    )
    train_fit = train_fit.reset_index(drop=True)
    val = val.reset_index(drop=True)

    # modalités conservées : au moins MIN_COUNT lignes dans train_fit
    keep = {}
    for c in CAT_COLS:
        vc = train_fit[c].value_counts()
        keep[c] = vc[vc >= MIN_COUNT].index.tolist()

    def group_rare(df):
        df = df.copy()
        for c in CAT_COLS:
            df[c] = df[c].where(df[c].isin(keep[c]), 'autres')
        return df

    train_fit, val, test = group_rare(train_fit), group_rare(val), group_rare(test)

    train_enc = _encode(train_fit)
    val_enc = _encode(val).reindex(columns=train_enc.columns, fill_value=0)
    test_enc = _encode(test).reindex(columns=train_enc.columns, fill_value=0)

    x_cols = [c for c in train_enc.columns if c not in (TARGET, 'attack_cat')]
    return train_enc, val_enc, test_enc, x_cols


def fingerprint(df):
    h = pd.util.hash_pandas_object(df, index=False).values
    return hashlib.md5(h.tobytes()).hexdigest()[:12]


def verify(train, val, test):
    """Compare les empreintes à celles du notebook 02_preparation."""
    got = {'train': fingerprint(train), 'val': fingerprint(val), 'test': fingerprint(test)}
    for k in got:
        statut = 'OK' if got[k] == EXPECTED[k] else 'DIFFÉRENTE'
        print(f'{k:5s} attendue {EXPECTED[k]} | obtenue {got[k]} | {statut}')
    return got == EXPECTED

In [7]:
# Chargement des données préparées et contrôle d'intégrité
train, val, test, x_cols = load_prepared()
verify(train, val, test)

# Le jeu de test est supprimé de la mémoire : on ne s'en sert qu'à la fin du projet
del test

train attendue fb8d73b1cdba | obtenue fb8d73b1cdba | OK
val   attendue f5db41df66c1 | obtenue f5db41df66c1 | OK
test  attendue 0f96e3a31bcc | obtenue 0f96e3a31bcc | OK


In [8]:
import os
import time
import datetime

import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (recall_score, precision_score, f1_score,
                             average_precision_score, matthews_corrcoef)

SEED = 42

# X = les caractéristiques, y = la réponse à prédire (0 normal, 1 attaque)
X_train = train[x_cols].astype("float64")
y_train = train["label"]
X_val = val[x_cols].astype("float64")
y_val = val["label"]

print(X_train.shape, X_val.shape)
print("Cœurs CPU disponibles :", os.cpu_count())

(80648, 58) (20163, 58)
Cœurs CPU disponibles : 4


In [10]:
modeles = {
    "regression_logistique": (
        make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=1000, random_state=SEED)),
        "défaut sauf max_iter=1000 ; normalisation dans le pipeline"),
    "arbre_decision": (
        DecisionTreeClassifier(random_state=SEED),
        "défaut"),
    "random_forest": (
        RandomForestClassifier(random_state=SEED, n_jobs=-1),
        "défaut"),
    "xgboost": (
        XGBClassifier(random_state=SEED, n_jobs=-1),
        "défaut"),
    "lightgbm": (
        LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1),
        "défaut"),
}

In [11]:
def evaluer(nom, modele, note):
    debut = time.perf_counter()
    modele.fit(X_train, y_train)
    temps = time.perf_counter() - debut

    pred = modele.predict(X_val)
    proba = modele.predict_proba(X_val)[:, 1]

    return {
        "date": datetime.date.today().isoformat(),
        "auteure": "Imene",
        "modele": nom,
        "methode": "defaut",
        "n_essais": "NA",
        "graine": SEED,
        "hyperparametres": note,
        "rappel": recall_score(y_val, pred),
        "precision": precision_score(y_val, pred),
        "f1": f1_score(y_val, pred),
        "pr_auc": average_precision_score(y_val, proba),
        "mcc": matthews_corrcoef(y_val, pred),
        "f1_train": f1_score(y_train, modele.predict(X_train)),
        "temps_entrainement_s": round(temps, 2),
    }

os.makedirs("results", exist_ok=True)
lignes = []
for nom, (modele, note) in modeles.items():
    ligne = evaluer(nom, modele, note)
    pd.DataFrame([ligne]).to_csv(f"results/baseline_{nom}.csv", index=False)
    lignes.append(ligne)
    print(nom, "terminé en", ligne["temps_entrainement_s"], "s")

regression_logistique terminé en 4.29 s
arbre_decision terminé en 2.7 s
random_forest terminé en 8.52 s
xgboost terminé en 1.29 s
lightgbm terminé en 1.25 s


In [12]:
tableau = pd.DataFrame(lignes).set_index("modele")
tableau[["rappel", "precision", "f1", "pr_auc", "mcc",
         "f1_train", "temps_entrainement_s"]].round(4)

,rappel,precision,f1,pr_auc,mcc,f1_train,temps_entrainement_s
modele,,,,,,,
regression_logistique,0.9759,0.8383,0.9019,0.9576,0.8040,0.9034,4.29
arbre_decision,0.9100,0.9094,0.9097,0.8714,0.8237,1.0000,2.70
random_forest,0.9488,0.9138,0.9310,0.9824,0.8635,1.0000,8.52
xgboost,0.9448,0.9172,0.9308,0.9839,0.8634,0.9546,1.29
lightgbm,0.9472,0.9127,0.9296,0.9832,0.8608,0.9400,1.25
